# 1111 49일차

## 1. 요청 처리량

1초에 요청을 몇 개 끝냈는지 재는 것 (`f24_single_request` ~ `f28_concurrency_compare`)

출력 토큰 수로 낸 TPS는 토큰을 만드는 속도라서 요청이 몇 개 끝났는지는 알 수 없음. 요청 단위로도 재야 동시에 들어오는 요청을 얼마나 받아 내는지 보임

1. `len(prompts) / elapsed` : 초당 요청 수(RPS)
2. `total_tokens / elapsed` : 초당 출력 토큰 수(TPS)
3. `for prompt in prompts: llm.invoke(prompt)` : 순차 요청. 앞 요청이 끝나야 다음이 나감
4. `executer.map(ask, prompts)` : 동시 요청. `max_workers`만큼 한꺼번에 보냄
5. `for workers in [1, 2, 4, 8]:` : 동시 요청 수만 바꿔 같은 측정을 반복함

- 순차로 보내면 RPS가 요청 하나의 지연에 묶여서 더 올라가지 않음
- 동시 요청 수를 늘리면 RPS와 TPS가 같이 올라감
- **`max_tokens`와 질문 수를 고정해야 동시 요청 수만의 차이를 봄**
- 동시 요청 수를 두 배로 올려도 RPS가 두 배가 되지는 않음
- 요청 하나의 지연과 전체 처리량은 다른 값. 처리량이 올라도 요청 하나는 느려질 수 있음

## 2. 요청 대기

동시 요청 수보다 요청이 많을 때 뒤 요청이 기다리는 것 (`f21_continuous_batching3`)

동시 요청 수를 늘려도 서버와 스레드가 한 번에 받는 양은 정해져 있음. 넘치는 요청이 언제 시작하는지 보려고 요청을 하나씩 따로 맡김

1. `pool.submit(ask, "A", 0, 800)` : 함수와 인자를 넘겨 요청 하나를 맡김
2. `futures = [...]` : 맡긴 결과를 받을 객체를 목록으로 모음
3. `[f.result() for f in futures]` : 요청이 끝날 때까지 기다려 결과를 꺼냄
4. `time.sleep(delay)` : 요청을 보내는 시점을 어긋나게 둠

- `submit`은 인자를 그대로 넘겨서 `map`과 달리 인자가 여러 개여도 됨
- `max_workers=4`에 요청이 5개면 다섯 번째는 앞 요청 하나가 끝난 뒤에 시작함
- 출력 상한이 작은 요청이 먼저 끝나서 시작 순서와 끝나는 순서가 다름
- `with` 블록을 벗어날 때 `shutdown()`이 자동으로 불림
- `print`에 `flush=True`를 둬야 시작·완료 순서가 밀리지 않고 그때그때 찍힘

## 3. 적중률 계산

같은 앞부분을 다시 보냈을 때 재사용된 비율을 이번 요청만으로 내는 것 (`f23_prefix_cache1`, `f23_prefix_cache2`)

`/metrics`의 조회·적중 횟수는 서버가 뜬 뒤로 계속 쌓인 값임. 그대로 읽으면 이번 요청의 적중률이 아니라 지금까지의 평균이 나옴

1. `prefix = "NPU 추론 최적화에 대한 설명입니다." * 150` : 앞부분이 같은 질문을 만듦
2. `before = get_cache_metrics()` : 요청 전 수치를 읽어 둠
3. `after = get_cache_metrics()` : 요청 뒤 수치를 다시 읽음
4. `hits = after["hits"] - before["hits"]` : 이번 요청에서 늘어난 값만 꺼냄

- **쌓이는 값은 전후 차이로 계산해야 이번 요청만의 적중률이 나옴**
- 앞부분이 같은 요청을 반복하면 두 번째부터 응답 시간이 조금 줄어듦
- 적중률이 올라가도 입력 토큰 수는 그대로 찍힘. 계산을 건너뛰는 것이고 토큰을 줄이는 게 아님
- 앞부분이 그대로 걸리면 이번 요청의 조회가 거의 전부 적중으로 잡힘

## 4. 서버 상태 확인

`/metrics`에서 보고 싶은 줄만 골라 읽는 것 (`f29_check_server_metric`)

메트릭은 수백 줄로 와서 다 찍으면 읽을 수 없음. 이름으로 걸러야 지금 돌고 있는 요청 수처럼 필요한 값만 봄

1. `requests.get(url, timeout=5)` : 주소를 열어 본문을 받음
2. `response.raise_for_status()` : 응답 코드가 정상이 아니면 에러를 냄
3. `keywords = [...]`와 `any(word in line for word in keywords)` : 키워드 중 하나라도 들어 있는 줄만 남김
4. `num_requests_running`·`num_requests_waiting` : 지금 처리 중인 요청 수와 대기 중인 요청 수

- `urllib.request`는 표준 라이브러리, `requests`는 설치해서 쓰는 패키지. 같은 주소를 읽음
- `raise_for_status()`를 두면 잘못된 응답을 그냥 지나치지 않음
- 히스토그램으로 오는 값은 `_bucket`·`_count`·`_sum`으로 쪼개져서, 이름만 걸면 구간마다 줄이 나옴
- `#`로 시작하는 줄은 이름 설명이라 건너뜀